<table>
  <tr>
    <td><div align="left"><font size="30">Using Torch with Machine Vision Toolbox for Python</font></div></td>
    <td><img src="https://github.com/petercorke/machinevision-toolbox-python/raw/main/docs/figs/VisionToolboxLogo_NoBackgnd@2x.png" width="400" height="317" style="height:auto"></td>
  </tr>
</table>

A quick, runnable introduction to the toolbox. Works locally, on Google Colab, or entirely in your browser via [JupyterLite](https://jupyterlite.readthedocs.io) and [Pyodide](https://pyodide.org) -- no installation required either way. The cell below installs the toolbox the first time it is run (and will take a few seconds on Colab or in the browser).

## Configuring the Jupyter environment
We need to import some packages to help us with linear algebra (`numpy`), graphics (`matplotlib`), and machine vision (`machinevisiontoolbox`). If you're running locally you need to have these packages installed.

<div class="alert alert-info">
If you're running on CoLab or in the browser we have to first install machinevisiontoolbox, which is not preinstalled, so the next cell will be slow.

<b>This notebook cannot be run in the browser.</b> torch and torchvision are not pure Python packages and are not included in Pyodide's built-in packages.
</div>

In [ ]:
# MVTB_BOOTSTRAP_CELL: installs the package, which takes a short time, so wait for the line "Running ..." below.  Generated by sync_bootstrap.py from _mvtb_nb_bootstrap.py, do not edit

import subprocess
import sys
PACKAGE = 'machinevision-toolbox-python'
IMPORT_NAME = 'machinevisiontoolbox'
GIT_URL: str | None = 'https://github.com/petercorke/machinevision-toolbox-python'
JL_PACKAGES = ['numpy', 'scipy', 'matplotlib', 'ipympl', 'pillow', 'typing_extensions', 'opencv-python', 'ansitable', 'mvtb-data', 'tqdm', 'requests', 'ipywidgets', 'spatialmath-python', 'pgraph-python']
JL_NO_DEPS: list[str] = []
JL_PACKAGE_NO_DEPS = True
JL_EXTRAS: dict[str, list[str]] = {}
COLAB_PACKAGES: list[str] = []

def _version_of(package: str, module) -> str:
    import importlib.metadata
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return str(getattr(module, '__version__', 'unknown'))

def _older(version: str, required: str) -> bool:
    try:
        from packaging.version import InvalidVersion, Version
        return Version(version) < Version(required)
    except (ImportError, InvalidVersion):
        return False

async def ensure_installed(extras: tuple[str, ...] | list[str]=(), packages: tuple[str, ...] | list[str]=(), min_version: str | None=None) -> bool:
    if sys.platform == 'emscripten':
        try:
            import piplite
            install = piplite.install
        except ImportError:
            import micropip
            install = micropip.install
        print(f'Installing {PACKAGE}, this takes a short time...', flush=True)
        for name in JL_NO_DEPS:
            await install(name, deps=False)
        for name in JL_PACKAGES:
            await install(name)
        for extra in extras:
            for name in JL_EXTRAS.get(extra, []):
                await install(name)
        for name in packages:
            await install(name)
        await install(PACKAGE, deps=not JL_PACKAGE_NO_DEPS)
        where, colab = ('in browser', False)
    else:
        try:
            import google.colab
        except ImportError:
            where, colab = ('locally', False)
        else:
            print(f'Installing {PACKAGE}, this takes a short time...', flush=True)
            requirement = PACKAGE
            if extras:
                requirement += '[' + ','.join(extras) + ']'
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', requirement] + list(packages) + COLAB_PACKAGES, check=True)
            try:
                from matplotlib.backends import backend_registry
            except ImportError:
                pass
            else:
                clear = getattr(backend_registry, '_clear', None)
                if clear is not None:
                    clear()
            from google.colab import output
            output.enable_custom_widget_manager()
            where, colab = ('on Colab', True)
    import importlib
    module = importlib.import_module(IMPORT_NAME)
    version = _version_of(PACKAGE, module)
    if colab and min_version is not None and _older(version, min_version) and GIT_URL:
        print(f'{PACKAGE} {version} on PyPI is older than {min_version}, installing from GitHub...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', f'{PACKAGE} @ git+{GIT_URL}@main'], check=True)
        importlib.invalidate_caches()
        version = _version_of(PACKAGE, module)
        where += ', development version from GitHub'
    if not colab and where == 'locally':
        import importlib.metadata
        for name in packages:
            try:
                importlib.metadata.version(name)
            except importlib.metadata.PackageNotFoundError:
                print(f'This notebook needs {name}: pip install {name}')
    if min_version is not None and where in ('locally', 'on Colab') and _older(version, min_version):
        print(f'This notebook needs {PACKAGE} {min_version} or later: pip install -U {PACKAGE}')
    print(f'Running {where} using {PACKAGE} v{version}')
    return colab

COLAB = await ensure_installed()


# Setting up


You must have TorchVision installed

```shell
$ pip install torchvision
$ pip install torch
```

or 

```shell
pip install machinevision-toolbox-python[torch]
```

Next, we import the packages we will need: `machinevisiontoolbox`, `matplotlib`, and `torchvision`.

In [ ]:
%matplotlib ipympl
import matplotlib.pyplot as plt
plt.close("all")   # re-running the notebook from the top starts with no figures

from machinevisiontoolbox import Image
import torchvision as tv

We will use the complex scene given as Fig 12.7 of Robotics, Vision & Control, 3rd edition, p. 492.

In [ ]:
scene = Image.Read("image3.jpg")
scene.disp()

Each object comprises a range of pixel intensities and colors so simple pixel-level algorithms cannot work.  Instead, we will use ResNet50, a 50-layer convolutional neural network for computer vision that uses residual "skip connections" to perform semantic segmentation.

We use a pretrained ResNet50 model from torchvision which was trained on a subset of the [COCO 2017 dataset](https://huggingface.co/datasets/rafaelpadilla/coco2017) using only the 20 object categories that overlap with [Pascal VOC dataset](https://www.robots.ox.ac.uk/~vgg/projects/pascal/VOC/), plus a background class 21 classes in all.

In [ ]:
model = tv.models.segmentation.fcn_resnet50(
    weights=tv.models.segmentation.FCN_ResNet50_Weights.COCO_WITH_VOC_LABELS_V1
).eval()

A `machinevisiontoolbox` image must be transformed before it can be used by the ResNet50 model.  It must first be converted to a "tensor" and then normalized so that its mean is zero and its standard deviation is one.

In [ ]:
in_tensor = scene.tensor(normalize="imagenet")
print("Input tensor type:", type(in_tensor))
print("Input tensor shape:", in_tensor.shape)

The tensor is a C++ object that behaves somewhat like a NumPy array.  The dimensions of this one are: number of images in the sequence, number of planes per image, image height, image width.

The tensor is passed to the model and the result is a `dict`

In [ ]:
outputs = model(in_tensor)
print("type(outputs):", type(outputs))

and the elements of that `dict` are tensors

In [ ]:
for item, value in outputs.items():
    print(f"Type of {item}: {type(value)}")

In [ ]:
print("Shape of outputs[\"out\"]:", outputs["out"].shape)

The inference result is essentially a 21-plane image with one plane per class.  For every pixel the index of the plane with the highest value is the class label of that pixel.

The `machinevisiontoolbox` can convert the inference result (a dict of tensors) into a class label image

In [ ]:
seg = Image.Tensor(outputs, logits=True)
seg.disp();

We can see that different objects in the scene appear as different grey values.  Exploring those values with the cursor we see that people correspond to a value of 15, and dogs to the value 12.

These are the class labels that indicate the semantic meaning of each pixel.  The network we used was trained on the Pascal VOC dataset which defines the following classes


In [ ]:
class_names = [
    "Background",
    "Aeroplane",
    "Bicycle",
    "Bird",
    "Boat",
    "Bottle",
    "Bus",
    "Car",
    "Cat",
    "Chair",
    "Cow",
    "Dining Table",
    "Dog",
    "Horse",
    "Motorbike",
    "Person",
    "Potted Plant",
    "Sheep",
    "Sofa",
    "Train",
    "TV/Monitor",
]

Using this list we can map class labels to class names

In [ ]:
print(class_names[15])
print(class_names[12])

With a bit more effort we can display this with class labels as colors, and a colorbar showing the mapping from class index to color

In [ ]:
seg.disp(
    colormap="viridis",
    ncolors=20,
    colorbar=dict(xticks=[range(21), class_names]),
);


# YOLO bounding boxes

This example demonstrates how to use a pretrained object detection model to
detect objects in an image and display the results with bounding boxes and class
labels.  

We will perform object detection using the well-known YOLO model, in particular we will use
the latest YOLO26 nano model from the ultralytics library. This model is very fast and
avoids using Non-Maximum Suppression (NMS) to reject overlapping boxes, instead it natively outputs a single box per object.

We first need to install the ultralytics library which provides a PyTorch implementation of the YOLO object detection model


```shell
$ pip install ultralytics
```
or 

```shell
pip install machinevision-toolbox-python[ultralytics]
```

We first import `YOLO`

In [ ]:
from ultralytics import YOLO

The model needs the neural weights and these are downloaded to your current folder.

In [ ]:
model = YOLO("yolo26n.pt")

and then run inference on the image as a Numpy array.  We do not have to do the normalization or conversion to a tensor as we did with the ResNet50 model earlier.

In [ ]:
results = model(scene.array)

The return value `results` is a list of results (one per image, but we only have one image here), and the results are packed into a `Results` instance.

In [ ]:
print("Length of results:", len(results))
print("Type of results[0]:", type(results[0]))

The results are still in GPU memory and we need to get the data back onto the CPU and convert to Numpy arrays so that we can work with them in Python

In [ ]:
boxes = results[0].boxes.xyxy.cpu().numpy()  # [xmin, ymin, xmax, ymax]
classes = results[0].boxes.cls.cpu().numpy()

print(boxes)
print(classes)

We see that `boxes` is a 2d-array with one row per box organized as `[xmin, ymin, xmax, ymax]`, and `classes` is a 1d-array with one class label per box.

Now we will overlay the boxes and the class labels on the image

In [ ]:
from machinevisiontoolbox.base import plot_labelbox
scene.disp()
for box, class_id in zip(boxes, classes):
    plot_labelbox(f"{model.names[class_id]}", lbrt=box, color="red")